%pip install optbinning pandas xlrd pandarallel ipywidgets tqdm

In [57]:
import pandas as pd
from pandarallel import pandarallel
from tqdm.notebook import tqdm

tqdm.pandas()


pandarallel.initialize(progress_bar=True)


INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [58]:
data = pd.read_excel(r"/workspaces/Strategic_Segment_Builder/default of credit card clients.xls", header = 1)

In [59]:
data.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [60]:
data.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='str')

In [61]:
from optbinning import OptimalBinning


In [62]:
data['default payment next month'].value_counts(normalize = True)*100

default payment next month
0    77.88
1    22.12
Name: proportion, dtype: float64

In [63]:
target = "default payment next month"

results = []

for col in data.columns:

    if col == target:
        continue

    try:

        dtype = (
            "categorical"
            if str(data[col].dtype) in ["object", "category"]
            else "numerical"
        )

        optb = OptimalBinning(
            name=col,
            dtype=dtype, 
            solver='cp'
        )

        optb.fit(
            data[col].values,
            data[target]
        )

        iv = list(optb.binning_table.build().IV)[-1]

        results.append({
            "variable": col,
            "iv": iv*100
        })

    except Exception as e:
        print(f"Skipping {col}: {e}")

iv_ranking = (
    pd.DataFrame(results)
      .sort_values("iv", ascending=False)
)

print(iv_ranking.head(20))

     variable         iv
6       PAY_0  86.938084
7       PAY_2  54.685883
8       PAY_3  41.242008
9       PAY_4  35.928931
10      PAY_5  33.373381
11      PAY_6  28.516449
18   PAY_AMT1  18.475602
1   LIMIT_BAL  17.983607
19   PAY_AMT2  16.504107
20   PAY_AMT3  13.207401
21   PAY_AMT4  11.124476
23   PAY_AMT6   9.852584
22   PAY_AMT5   9.285231
5         AGE   2.220189
3   EDUCATION   1.599044
16  BILL_AMT5   1.583242
17  BILL_AMT6   1.417074
15  BILL_AMT4   1.184494
0          ID   1.057210
12  BILL_AMT1   0.952468


In [64]:
top_vars = iv_ranking.variable[0:6]

In [65]:


binned_df = pd.DataFrame()

for col in top_vars:

    dtype = (
        "categorical"
        if str(data[col].dtype) in ["object", "category"]
        else "numerical"
    )

    optb = OptimalBinning(
        name=col,
        dtype=dtype
    )

    optb.fit(
        data[col].values,
        data[target]
    )

    binned_df[col] = optb.transform(
        data[col],
        metric="bins"
    )

binned_df[target] = data[target]

In [66]:
binned_df.head()

,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,default payment next month
0,"[1.50, inf)","[1.50, inf)","[-1.50, -0.50)","[-1.50, -0.50)","(-inf, -1.50)","(-inf, -1.50)",1
1,"(-inf, -0.50)","[1.50, inf)","[-0.50, 1.50)","[-0.50, 0.50)","[-0.50, 1.00)","[1.00, inf)",1
2,"[-0.50, 0.50)","[-0.50, 1.50)","[-0.50, 1.50)","[-0.50, 0.50)","[-0.50, 1.00)","[-0.50, 1.00)",0
3,"[-0.50, 0.50)","[-0.50, 1.50)","[-0.50, 1.50)","[-0.50, 0.50)","[-0.50, 1.00)","[-0.50, 1.00)",0
4,"(-inf, -0.50)","[-0.50, 1.50)","[-1.50, -0.50)","[-0.50, 0.50)","[-0.50, 1.00)","[-0.50, 1.00)",0


In [67]:
base_rate = data['default payment next month'].mean()

In [68]:
base_rate

np.float64(0.2212)

In [69]:
results = []

for col in binned_df.columns:

    if col == target:
        continue

    summary = (
        binned_df
        .groupby(col)
        .agg(
            count=(target, "size"),
            events=(target, "sum")
        )
        .reset_index()
    )

    summary["rate"] = (
        summary["events"] /
        summary["count"]
    )*100

    summary["lift"] = (
        summary["rate"] /
        base_rate
    )

    summary["rule"] = (
        col + "=" +
        summary[col].astype(str)
    )

    results.append(
        summary[
            ["rule","count","rate","lift"]
        ]
    )

one_way = pd.concat(results)

In [70]:
one_way.sort_values(["lift","rate","count"], ascending=False)

,rule,count,rate,lift
3,"PAY_0=[1.50, inf)",3130,69.552716,314.433615
3,"PAY_2=[1.50, inf)",4410,56.031746,253.308074
3,"PAY_5=[1.00, inf)",2968,55.559299,251.172239
3,"PAY_4=[0.50, inf)",3510,53.532764,242.010685
3,"PAY_6=[1.00, inf)",3079,52.322183,236.537896
3,"PAY_3=[1.50, inf)",4209,52.292706,236.404639
2,"PAY_0=[0.50, 1.50)",3688,33.947939,153.471696
0,"PAY_6=(-inf, -1.50)",4895,20.040858,90.600624
0,"PAY_5=(-inf, -1.50)",4546,19.687637,89.003786
0,"PAY_4=(-inf, -1.50)",4348,19.250230,87.026356


In [71]:
def shortlist_rules(count,lift,rate, min_sample_size = 1000, min_lift = 200):
    if count>=min_sample_size and lift>=min_lift and rate>base_rate*100:
        return 1
    else:
        return 0

    

In [72]:
one_way['shortlist'] = one_way.parallel_apply(lambda x: shortlist_rules(x['count'],x['lift'], x['rate']), axis = 1)
one_way

,rule,count,rate,lift,shortlist
0,"PAY_0=(-inf, -0.50)",8445,15.618709,70.608993,0
1,"PAY_0=[-0.50, 0.50)",14737,12.811291,57.917230,0
2,"PAY_0=[0.50, 1.50)",3688,33.947939,153.471696,0
3,"PAY_0=[1.50, inf)",3130,69.552716,314.433615,1
0,"PAY_2=(-inf, -1.50)",3782,18.270756,82.598355,0
1,"PAY_2=[-0.50, 1.50)",15758,15.915725,71.951742,0
2,"PAY_2=[-1.50, -0.50)",6050,15.966942,72.183283,0
3,"PAY_2=[1.50, inf)",4410,56.031746,253.308074,1
0,"PAY_3=(-inf, -1.50)",4085,18.531212,83.775822,0
1,"PAY_3=[-0.50, 1.50)",15768,17.453070,78.901761,0


In [73]:
from itertools import combinations

results = []

cols = [
    c for c in binned_df.columns
    if c != target
]

for c1, c2 in combinations(cols, 2):

    summary = (
        binned_df
        .groupby([c1, c2])
        .agg(
            count=(target, "size"),
            events=(target, "sum")
        )
        .reset_index()
    )

    summary["rate"] = (
        summary["events"] /
        summary["count"]
    )*100

    summary["lift"] = (
        summary["rate"] /
        base_rate
    )

    summary["rule"] = (
        c1 + "=" +
        summary[c1].astype(str)
        + " & " +
        c2 + "=" +
        summary[c2].astype(str)
    )

    results.append(
        summary[
            ["rule","count","rate","lift"]
        ]
    )

two_way = pd.concat(results)

In [74]:
two_way = two_way.sort_values(["lift","rate","count"], ascending=False)
two_way['shortlist'] = two_way.parallel_apply(lambda x: shortlist_rules(x['count'],x['lift'], x['rate']), axis = 1)
two_way

,rule,count,rate,lift,shortlist
1,"PAY_2=(-inf, -1.50) & PAY_3=[-0.50, 1.50)",1,100.000000,452.079566,0
15,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf)",1249,76.621297,346.389227,1
15,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf)",1309,76.088617,343.981091,1
15,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf)",1496,74.264706,335.735560,1
15,"PAY_0=[1.50, inf) & PAY_3=[1.50, inf)",1754,72.862030,329.394347,1
...,...,...,...,...,...
6,"PAY_0=[-0.50, 0.50) & PAY_3=[-1.50, -0.50)",763,9.829620,44.437703,0
1,"PAY_4=(-inf, -1.50) & PAY_6=[-0.50, 1.00)",115,8.695652,39.311267,0
3,"PAY_2=(-inf, -1.50) & PAY_3=[1.50, inf)",3,0.000000,0.000000,0
1,"PAY_4=(-inf, -1.50) & PAY_5=[-0.50, 1.00)",3,0.000000,0.000000,0


In [75]:
results = []

for c1, c2, c3 in combinations(cols, 3):

    summary = (
        binned_df
        .groupby([c1, c2, c3])
        .agg(
            count=(target, "size"),
            events=(target, "sum")
        )
        .reset_index()
    )

    summary["rate"] = (
        summary["events"] /
        summary["count"]
    )*100

    summary["lift"] = (
        summary["rate"] /
        base_rate
    )

    summary["rule"] = (
        c1 + "=" +
        summary[c1].astype(str)
        + " & " +
        c2 + "=" +
        summary[c2].astype(str)
        + " & " +
        c3 + "=" +
        summary[c3].astype(str)
    )

    results.append(
        summary[
            ["rule","count","rate","lift"]
        ]
    )

three_way = pd.concat(results)

In [76]:

three_way['shortlist'] = three_way.parallel_apply(lambda x: shortlist_rules(x['count'],x['lift'], x['rate']), axis = 1)
three_way = three_way.sort_values(["lift","rate","count"], ascending=False)

In [77]:
three_way_shortlisted = three_way.loc[three_way.shortlist == 1,:]
two_way_shortlisted = two_way.loc[two_way.shortlist == 1,:]
one_way_shortlisted = one_way.loc[two_way.shortlist == 1,:]


In [78]:
all_shortlisted = pd.concat([one_way_shortlisted.reset_index(drop= True),two_way_shortlisted.reset_index(drop= True),three_way_shortlisted.reset_index(drop= True)])

In [84]:
all_shortlisted = all_shortlisted.sort_values(["lift","rate","count"], ascending=False).reset_index()

In [96]:
all_shortlisted

,index,rule,count,rate,lift,shortlist
0,0,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_6=...",1002,77.245509,349.211162,1
1,1,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf) & PAY_6=...",1072,77.238806,349.180859,1
2,2,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf) & PAY_6=...",1091,77.176902,348.901003,1
3,3,"PAY_0=[1.50, inf) & PAY_3=[1.50, inf) & PAY_6=...",1024,76.855469,347.447870,1
4,0,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf)",1249,76.621297,346.389227,1
5,4,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_5=...",1181,76.291279,344.897281,1
6,1,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf)",1309,76.088617,343.981091,1
7,5,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf) & PAY_5=...",1159,75.841242,342.862760,1
8,6,"PAY_0=[1.50, inf) & PAY_3=[1.50, inf) & PAY_5=...",1108,75.722022,342.323787,1
9,2,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf)",1496,74.264706,335.735560,1


In [87]:
first_rule = all_shortlisted.loc[0,'rule']

In [88]:
first_rule

'PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_6=[1.00, inf)'

In [90]:
import re
import numpy as np
import pandas as pd

def parse_rule(rule_str):
    parts = [p.strip() for p in rule_str.split('&')]
    conditions = []
    for p in parts:
        col, interval = p.split('=', 1)
        col = col.strip()
        interval = interval.strip()
        if len(interval) < 2:
            raise ValueError(interval)
        include_lower = interval[0] == '['
        include_upper = interval[-1] == ']'
        inner = interval[1:-1].strip()
        lower_str, upper_str = [s.strip() for s in inner.split(',', 1)]
        def to_bound(s):
            s = s.lower()
            if s in ('inf', 'infty', '+inf', '+infty'):
                return np.inf
            if s in ('-inf', '-infty'):
                return -np.inf
            return float(s)
        lower = to_bound(lower_str)
        upper = to_bound(upper_str)
        conditions.append({
            'col': col,
            'lower': lower,
            'upper': upper,
            'include_lower': include_lower,
            'include_upper': include_upper,
        })
    return conditions

def build_mask(df, conditions):
    mask = pd.Series(True, index=df.index)
    for c in conditions:
        col = c['col']
        if col not in df.columns:
            raise KeyError(f"Column {col} not in DataFrame")
        # ensure numeric for comparisons
        col_series = pd.to_numeric(df[col], errors='coerce')
        cm = pd.Series(True, index=df.index)
        if not np.isneginf(c['lower']) and not np.isposinf(c['lower']):
            if c['include_lower']:
                cm &= col_series >= c['lower']
            else:
                cm &= col_series > c['lower']
        if not np.isposinf(c['upper']) and not np.isneginf(c['upper']):
            if c['include_upper']:
                cm &= col_series <= c['upper']
            else:
                cm &= col_series < c['upper']
        mask &= cm.fillna(False)
    return mask

# Usage:
# conditions = parse_rule(first_rule)
# mask = build_mask(df, conditions)
# df_filtered = df[mask]

In [101]:
conditions = parse_rule(first_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]

In [102]:
df_filtered['default payment next month'].value_counts(normalize = True)*100

default payment next month
0    79.784813
1    20.215187
Name: proportion, dtype: float64

In [103]:
import pandas as pd

# --- Helper: implement or reuse existing pipeline ---
def run_pipeline(df):
    """
    Run your existing pipeline on `df` and return:
      one_way_shortlisted, two_way_shortlisted, three_way_shortlisted

    Replace the placeholder below with the code from your notebook
    that produces these three DataFrames.
    Each returned DataFrame must contain at least these columns:
      - 'rule' (string)
      - 'lift' (numeric)
      - 'rate' (numeric)
      - 'count' (numeric)

    Example return:
      return one_way_shortlisted, two_way_shortlisted, three_way_shortlisted
    """
    # ---------- BEGIN: paste your pipeline here ----------
    # Example (REMOVE / REPLACE with your actual pipeline):
    # global data, target
    # data = df
    # <run your binning + rule-generation cells, but reading from `data`>
    # ---------- END: paste your pipeline here ----------
    raise NotImplementedError("Fill run_pipeline(df) with your pipeline that returns the three shortlisted DataFrames")

# --- Loop that iteratively extracts rules and filters the dataset ---
def iterative_rule_extraction(df, max_iters=100):
    remaining = df.copy()
    rule_log = []                     # list of chosen rules (strings)
    log_details = []                  # list of dicts with metadata per iteration

    for it in range(max_iters):
        if remaining.empty:
            break

        # Run your pipeline on the remaining data
        one_way_shortlisted, two_way_shortlisted, three_way_shortlisted = run_pipeline(remaining)

        # Combine and rank exactly as in your notebook
        all_shortlisted = pd.concat([
            one_way_shortlisted.reset_index(drop=True),
            two_way_shortlisted.reset_index(drop=True),
            three_way_shortlisted.reset_index(drop=True)
        ], ignore_index=True)

        if all_shortlisted.empty:
            break

        all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

        # Pick the top rule
        first_rule = all_shortlisted.loc[0, "rule"]

        # Record metadata
        top_meta = {
            "iteration": it + 1,
            "rule": first_rule,
            "lift": all_shortlisted.loc[0, "lift"],
            "rate": all_shortlisted.loc[0, "rate"],
            "count": int(all_shortlisted.loc[0, "count"]),
            "remaining_before": int(len(remaining))
        }
        rule_log.append(first_rule)
        log_details.append(top_meta)

        # Build mask for the rule (rows that MATCH the rule)
        conditions = parse_rule(first_rule)          # uses your existing helper
        mask = build_mask(remaining, conditions)     # uses your existing helper

        # Filter out the matched rows (apply anti-rule)
        remaining = remaining.loc[~mask].copy()

        # Stopping if nothing left
        if remaining.empty:
            break

    return rule_log, pd.DataFrame(log_details), remaining

# --- Usage example ---
rule_log, log_df, remaining_df = iterative_rule_extraction(data)
log_df  # view iteration metadata

NotImplementedError: Fill run_pipeline(df) with your pipeline that returns the three shortlisted DataFrames